# # QLoRA fine-tuning of Qwen 2.5 7B Instruct on our French cardiology dataset (Google Colab, free T4 GPU)
#
# **Why Colab?** Fine-tuning a 7B model needs a GPU with ~15 GB of memory: my PC has no such GPU
# (TensorFlow has no GPU on Windows, and my GTX 1060 has only 6 GB). Colab gives a free **T4 (16 GB)**.
#
# **QLoRA in 3 lines:** the 7B model is loaded in **4-bit** (QLoRA = Quantized LoRA) and stays frozen; we only
# train small **LoRA** matrices (rank r = 16) added to the attention and feed-forward layers
# (q, k, v, o, gate, up, down). So only ~0.5 % of the weights learn, and it fits in the T4.
#
# ## Steps for Farah
# 1. Open this notebook in Colab (File → Upload notebook, or directly from GitHub:
#    `https://colab.research.google.com/github/Farah-Belghith2710/CardioAgent/blob/final-merge/llm_finetuning/train_qlora_colab.ipynb`),
#    then **Runtime → Change runtime type → T4 GPU → Save**.
# 2. The dataset `cardio_instructions.jsonl` (made by `python llm_finetuning/build_dataset.py`) is downloaded from the
#    GitHub repo by cell 6 (or upload your own copy in the left panel, folder icon).
# 3. **Runtime → Run all**. When asked, allow access to Google Drive (the results are copied there).
#    It takes about 40-60 min: install (~5 min), training (~15-25 min), GGUF export (~15 min).
# 4. At the end, in Google Drive, folder **`MyDrive/cardioagent_llm/`**, download:
#    - `trainer_state.json` and `run_info.json` → put them in `llm_finetuning/outputs/` (for the MLflow script)
#    - `cardio_lora_adapter.zip` → unzip it in `llm_finetuning/outputs/cardio_lora_adapter/`
#    - `cardio-qwen2.5-7b-q4_k_m.gguf` (~4.7 GB) → put it in `llm_server/models/`
# 5. On my PC: `venv\Scripts\python llm_finetuning\log_finetuning_to_mlflow.py` (loss curve → MLflow),
#    then restart the platform: the llm-server sees the GGUF and creates the model `cardio-llm`.
#
# If the free session stops (Colab gives ~1-3 h of GPU), run all again: the dataset is small, it restarts fast.

In [ ]:
# 1. Install Unsloth (it brings TRL, PEFT, bitsandbytes, Hugging Face transformers) - about 5 minutes
# If this cell fails, copy the install cell of an official Unsloth Colab notebook (https://docs.unsloth.ai).
!pip install -q unsloth

In [ ]:
# 2. Check that we have a GPU (it must say "Tesla T4")
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1024**3))

In [ ]:
# 3. The settings of the fine-tuning (also written in run_info.json, then logged in MLflow)
BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"    # Qwen 2.5 7B Instruct, already quantized in 4-bit
MAX_SEQ_LENGTH = 1024    # the longest example (instruction + audio JSON + answer) is ~700 tokens
LORA_R = 16              # the rank of the LoRA matrices (bigger = more weights that learn)
LORA_ALPHA = 16          # the scale of the LoRA update (alpha / r = 1)
LEARNING_RATE = 2e-4
BATCH_SIZE = 2           # examples per step on the GPU
GRAD_ACCUM = 4           # gradient accumulation: 2 x 4 = 8 examples per weight update
EPOCHS = 1
SEED = 42

# the role of the model (the same idea as the SYSTEM of the Modelfile)
SYSTEM_PROMPT = ("Tu es un assistant médical prudent spécialisé dans les sons du cœur (phonocardiogrammes). "
                 "Tu réponds en français, simplement. Tu expliques les résultats du moteur audio sans jamais poser "
                 "de diagnostic définitif ni prescrire de traitement, et tu recommandes toujours de consulter un "
                 "médecin. En cas de symptômes graves, tu demandes d'appeler les urgences.")

In [ ]:
# 4. Load the 7B model in 4-bit + its tokenizer
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,     # QLoRA: the frozen base model in 4-bit
    dtype=None,            # None = automatic (float16 on a T4)
)

In [ ]:
# 5. Add the LoRA matrices: only them will learn
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",    # saves GPU memory
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
# 6. The dataset: each example becomes a conversation in the chat format of Qwen (ChatML)
#    system (the role) -> user (instruction + input) -> assistant (the answer to learn)
import os
import urllib.request
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# the dataset: uploaded in the Files panel, or else downloaded from our GitHub repo (the same file)
DATASET_URL = ("https://raw.githubusercontent.com/Farah-Belghith2710/CardioAgent/final-merge/"
               "llm_finetuning/cardio_instructions.jsonl")
if not os.path.exists("cardio_instructions.jsonl"):
    urllib.request.urlretrieve(DATASET_URL, "cardio_instructions.jsonl")

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
dataset = load_dataset("json", data_files="cardio_instructions.jsonl", split="train")
print(dataset)


def to_chat_text(example):
    user_message = example["instruction"]
    if example["input"]:
        user_message = user_message + "\n\n" + example["input"]
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message},
                {"role": "assistant", "content": example["output"]}]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}


dataset = dataset.map(to_chat_text)
print(dataset[0]["text"][:1500])

In [ ]:
# 7. The trainer (TRL SFTTrainer = supervised fine-tuning): 1 epoch, 1 log line per step
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=1,          # 1 point of the loss curve per step
        seed=SEED,
        output_dir="outputs",
        report_to="none",         # the loss curve goes to MLflow later, from my PC
    ),
)
# the loss is computed only on the answers of the assistant (not on the system prompt or the question)
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")

In [ ]:
# 8. Train! (~15-25 min on a T4)
import time

torch.cuda.reset_peak_memory_stats()
start = time.time()
trainer_stats = trainer.train()
train_minutes = (time.time() - start) / 60

peak_vram_gb = torch.cuda.max_memory_reserved() / 1024**3
print("Training time: {:.1f} min".format(train_minutes))
print("Peak GPU memory (VRAM) used: {:.2f} GB".format(peak_vram_gb))    # for the report

In [ ]:
# 9. Save the training history (the loss of each step) + the settings, for log_finetuning_to_mlflow.py
import json

trainer.state.save_to_json("trainer_state.json")
run_info = {
    "base_model": BASE_MODEL, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS, "max_seq_length": MAX_SEQ_LENGTH, "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM, "load_in_4bit": True, "n_examples": len(dataset),
    "peak_vram_gb": round(peak_vram_gb, 2), "train_minutes": round(train_minutes, 1),
    "gpu": torch.cuda.get_device_name(0),
}
with open("run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)
print(run_info)

In [ ]:
# 10. A quick test: the fine-tuned model explains a result of the audio engine
FastLanguageModel.for_inference(model)
audio_result = {"duration_s": 12.4, "prediction": "abnormal", "probability_abnormal": 0.86, "model": "CNN",
                "warning": None}
messages = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Explique ce résultat du moteur audio à un patient, avec prudence.\n\n"
                                        + json.dumps(audio_result, ensure_ascii=False)}]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=300, temperature=0.2)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
# 11. Save the LoRA adapter (a few dozen MB): it is the "result" of the fine-tuning
import shutil

model.save_pretrained("cardio_lora_adapter")
tokenizer.save_pretrained("cardio_lora_adapter")
shutil.make_archive("cardio_lora_adapter", "zip", "cardio_lora_adapter")
print("cardio_lora_adapter.zip saved")

In [ ]:
# 12. Export to GGUF q4_k_m (4-bit) for Ollama (~15 min: Unsloth merges the adapter, then quantizes with llama.cpp)
import glob
import os

model.save_pretrained_gguf("cardio_gguf", tokenizer, quantization_method="q4_k_m")
gguf_files = glob.glob("cardio_gguf/*.gguf") + glob.glob("cardio_gguf*.gguf")    # the name depends on the version
print(gguf_files)
gguf_file = [f for f in gguf_files if "q4_k_m" in f.lower()][0]
shutil.move(gguf_file, "cardio-qwen2.5-7b-q4_k_m.gguf")
print("GGUF ready: {:.2f} GB".format(os.path.getsize("cardio-qwen2.5-7b-q4_k_m.gguf") / 1024**3))

In [ ]:
# 13. Copy the results to Google Drive (MyDrive/cardioagent_llm/), then download them from Drive
from google.colab import drive

drive.mount("/content/drive")
destination = "/content/drive/MyDrive/cardioagent_llm/"
os.makedirs(destination, exist_ok=True)
for file_name in ["trainer_state.json", "run_info.json", "cardio_lora_adapter.zip", "cardio-qwen2.5-7b-q4_k_m.gguf"]:
    shutil.copy(file_name, destination)
    print("copied:", file_name, "({:.1f} MB)".format(os.path.getsize(file_name) / 1024**2))